# Mandatory Assignment 2: Ensemble Models & Artificial Neural Networks
#### Group: 04
Group members: Brage Blomkvist, Patrick Rafael Jemieljanczyk & William Bjørling Nilsen

In [19]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder, FunctionTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


from sklearn.tree import DecisionTreeClassifier


## Description of Dataset

The dataset used is a set of daily weather observations from Australia. The observations are taken from a span of ten years, between 2007 and 2017. 49 different locations are used to source the data. Most of the locations have a similar number of data entries, but there are some slight differences. From the Kaggle page there seems to be most difference in entries for the periods of 2007-08 and late 2012 to early 2013.

The purpose of the dataset is to see which days it is going to rain on, using the previous day's weather observations. The target value is a binary yes and no, depicting whether it rained or not on a given next day. The target value is stored in the column: "RainTomorrow" Each instance represents a day of weather observations in a time series. Each instance has a value for whether it rained on the day of the observations, and whether it rained on the next day. The threshold for rain that has been used to determine the value of both of these columns is a value of 1mm rain inside the full 24 hours of a day.

The dataset is available on Kaggle through the site publisher Joe Young (https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package/data). The data was retrieved from the Australian Bureau of Meteorology and was collected daily from station observations. The dataset has missing values in multiple features and rows, and even the target value.

The dataset includes multiple guidelines that the developers utilizing the dataset should follow. Since this is a time series, chronological sampling of data should be used. Another guideline is to only fit preprocessing to training data. This will reduce the risk of leakage, since the values, such as mean, will only be derived from the training data itself. In other words, data based on test and validation partitions will not be fitted to the training data. The last guideline is to use more evaluation scores than just accuracy.

### Problem

In [20]:
# Read the dataset
dataset_AUS = pd.read_csv('Dataset/weatherAUS.csv')
# Uses existing column names baked into csv file.

# Show the first 5 rows to validate that the data import has been successful
dataset_AUS.head()



,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


## Data Processing

In [21]:
# Overview of dataset
# Look at the implementation details, data types and missing data
dataset_AUS.info()

<class 'pandas.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  str    
 1   Location       145460 non-null  str    
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  str    
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  str    
 10  WindDir3pm     141232 non-null  str    
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null   float64


In [22]:
# See how many different locations are used.
dataset_AUS.value_counts("Location").nunique()


10

In [23]:
# Check if every location has the same amount of observations
dataset_AUS.value_counts("Location")

Location
Canberra            3436
Sydney              3344
Melbourne           3193
Brisbane            3193
Adelaide            3193
Perth               3193
Hobart              3193
Darwin              3193
Albury              3040
Wollongong          3040
MountGinini         3040
Ballarat            3040
Bendigo             3040
Cairns              3040
GoldCoast           3040
Townsville          3040
MountGambier        3040
Albany              3040
Launceston          3040
AliceSprings        3040
Newcastle           3039
Penrith             3039
Tuggeranong         3039
BadgerysCreek       3009
Cobar               3009
CoffsHarbour        3009
Moree               3009
NorfolkIsland       3009
Richmond            3009
SydneyAirport       3009
WaggaWagga          3009
Williamtown         3009
Sale                3009
MelbourneAirport    3009
Mildura             3009
Portland            3009
Watsonia            3009
Dartmoor            3009
Nuriootpa           3009
Woomera         

### Missing Data

In [24]:
# Look at the amount of missing values
dataset_AUS.isna().sum()

Date                 0
Location             0
MinTemp           1485
MaxTemp           1261
Rainfall          3261
Evaporation      62790
Sunshine         69835
WindGustDir      10326
WindGustSpeed    10263
WindDir9am       10566
WindDir3pm        4228
WindSpeed9am      1767
WindSpeed3pm      3062
Humidity9am       2654
Humidity3pm       4507
Pressure9am      15065
Pressure3pm      15028
Cloud9am         55888
Cloud3pm         59358
Temp9am           1767
Temp3pm           3609
RainToday         3261
RainTomorrow      3267
dtype: int64

In [25]:
dataset_AUS.count()

Date             145460
Location         145460
MinTemp          143975
MaxTemp          144199
Rainfall         142199
Evaporation       82670
Sunshine          75625
WindGustDir      135134
WindGustSpeed    135197
WindDir9am       134894
WindDir3pm       141232
WindSpeed9am     143693
WindSpeed3pm     142398
Humidity9am      142806
Humidity3pm      140953
Pressure9am      130395
Pressure3pm      130432
Cloud9am          89572
Cloud3pm          86102
Temp9am          143693
Temp3pm          141851
RainToday        142199
RainTomorrow     142193
dtype: int64

### Splitting the Dataset
Created temporary splitting of data


In [26]:
# Gridsearch can be used, but Nested Cross validation should be considered first. Must use Cross validation for evaluation instead of random states.

# Creating a temporary dataset
dataset_AUS_temp = pd.read_csv('Dataset/weatherAUS.csv')

# Temporary separating into targets and features
y = dataset_AUS_temp['RainTomorrow']
x = dataset_AUS_temp.drop(columns=['RainTomorrow'])

# Temporary splitting
# Split into training, testing and validation
features_train, features_test, targets_train, targets_test = train_test_split(
    x, y,
    train_size=0.8,
    test_size=0.2,
)

features_train, features_validation, targets_train, targets_validation = train_test_split(
    features_train, targets_train,
    train_size=0.75,
    test_size=0.25,
)

# Select numerical columns
numerical_features = x.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

# Select categorical columns
categorical_features = x.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()

# Pipelines
to do:
   - Make the processor able to remove certain columns. Can then eventually see which models perform better

   - Can make it an option to delete the feature day_of_year

   - Make an option in the pipeline which can delete the following columns: temp3am, temp9am, year and pressure3pm, day (Need to say what Brage says bout this)


In [27]:
# Putting the features which need to be encoded into variables as a list. To be used by pipelines to encode
location_features = ['Location']

rain_today_features = ['RainToday']

wind_features = [
    'WindGustDir',
    'WindDir9am',
    'WindDir3pm'
]

date_features = ['Date']

In [28]:
# Cyclical encoder function
def cyclic_encoder(series, period):
    sin = np.sin(2 * np.pi * series / period)
    cos = np.cos(2 * np.pi * series / period)

    return sin, cos


In [29]:
wind_map = {
    'N': 1, 'NNE': 2, 'NE': 3, 'ENE': 4,
    'E': 5, 'ESE': 6, 'SE': 7, 'SSE': 8,
    'S': 9, 'SSW': 10, 'SW': 11, 'WSW': 12,
    'W': 13, 'WNW': 14, 'NW': 15, 'NNW': 16
}

WIND_DIRECTIONS = 16
MONTH_PERIOD = 12

In [30]:
def encode_rain_today(rain_today_dataframe):
    return rain_today_dataframe.replace({'No': 0, 'Yes': 1}).astype(float)

# encode_wind() function needs for-loop because there are three features which need to be encoded by this function.
def encode_wind(wind_data_frame):
    result = pd.DataFrame()

    for column in wind_data_frame.columns:

        wind_numbers = wind_data_frame[column].map(wind_map).astype(float)

        sin, cos = cyclic_encoder(wind_numbers, WIND_DIRECTIONS)
        result[column + '_Sine'], result[column + '_Cosine'] = sin, cos

    return result

# encode_date() function doesn't need a for loop because there is only a single feature with the dates provided.
def encode_date(dataframe):
    result = pd.DataFrame()

    date = pd.to_datetime(dataframe['Date'])
    year = date.dt.year
    month = date.dt.month
    day = date.dt.dayofyear

    is_leap_year = ((year % 4 == 0) & (year % 100 != 0)) | (year % 400 == 0)
    days_in_year = np.select([is_leap_year], [366], default = 365)

    result['Month_Sine'], result['Month_Cosine'] = cyclic_encoder(month, MONTH_PERIOD)

    result['Day_Sine'], result['Day_Cosine'] = cyclic_encoder(day, days_in_year)

    result['Year'] = year

    return result


### Encoding pipelines

In [31]:
# Numerical pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Location pipeline
location_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(target_type='binary',))
])

rain_today_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_rain_today)),
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# Wind pipeline
wind_cyclical_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_wind)),
    ('imputer', SimpleImputer(strategy='constant'))
])

# Date pipeline
date_cyclical_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_date)),
    ('imputer', SimpleImputer(strategy='median'))
])



### Preprocessor

In [32]:
# Preprocessor
preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('location', location_pipeline, location_features),
    ('rain_today', rain_today_pipeline, rain_today_features),
    ('wind', wind_cyclical_pipeline, wind_features),
    ('date', date_cyclical_pipeline, date_features)
])

In [33]:
# Decision tree pipeline
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', LogisticRegression(max_iter=500))
])

# K-nearest neighbours
knn_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', KNeighborsClassifier(
        n_neighbors=5
    ))
])

# train decision tree
#dt_pipe.fit(x_train, y_train)

### Normalization

### Encoding

In [40]:
# Check encoding
# dataset_AUS.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


## Testing Function for Automatic Tuning

In [34]:
def validation(model):
    train_predictions = model.predict(features_train)
    validation_predictions = model.predict(features_validation)
    print(f'Training Accuracy: {accuracy_score(targets_train["Grades"], train_predictions,):.2f} Validation: {accuracy_score(targets_validation["Grades"], validation_predictions):.2f}')
    print(f'Training Precision: {precision_score(targets_train["Grades"], train_predictions, average="micro"):.2f} Validation: {precision_score(targets_validation["Grades"], validation_predictions, average="micro"):.2f}')
    print(f'Training Recall: {recall_score(targets_train["Grades"], train_predictions, average="micro"):.2f} Validation: {recall_score(targets_validation["Grades"], validation_predictions, average="micro"):.2f}')
    print(f'Training F1: {f1_score(targets_train["Grades"], train_predictions, average="micro"):.2f} Validation: {f1_score(targets_validation["Grades"], validation_predictions, average="micro"):.2f}')

## Baseline Models

### Decision Trees
The decision tree baseline model is used with automated tuning.

In [35]:
# Set a flag for whether a new model is trained or an old is loaded.
dt_new_flag = 0

In [36]:
# Pipeline for decision tree
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])

For all the automated tuning, plots are used to see whether the range of parameters chosen is enough to cover the parameters which yields the best result. This process is started by using an initial broad range of parameters, as shown below. The process is only shown for this model type. For other models, only the last parameters and check is provided.

In [37]:
# Initial parameters for decision tree to check which are tuned for decision trees.
dt_params_init = {
    'max_depth': [2, 4, 8, 12],
    'max_features': [None, 'sqrt', 4, 6]
}

In [38]:


# fit initial model to investigate parameters.
dt_grid_init = GridSearchCV(dt_pipe, dt_params_init)
dt_grid_init.fit(features_train, targets_train)


TypeError: '<' not supported between instances of 'float' and 'str'

In [ ]:
# Validate tuning
validation(dt_grid_init)

In [ ]:
# Final set of parameters to check with GridSearchCV function.

In [ ]:
dt_params = {
    'max_depth': [2, 4, 8, 12],
    'max_features': [None, 'sqrt', 4, 6]
}

In [ ]:
# Fit model if either the flag for a new model is active or no model is found. If the flag is false and a model is found, the old model is loaded.
dt_grid = GridSearchCV(dt_pipe, dt_params)
dt_grid.fit(features_train, targets_train)

In [ ]:
dt_cv_results = pd.DataFrame(dt_grid.cv_results_)
dt_cv_results

In [ ]:
# Values of None are passed in as a string for the violin plot to work.
dt_cv_results['param_max_features'] = dt_cv_results['param_max_features'].fillna('None')
sns.violinplot(data=dt_cv_results, x='param_max_features', y='mean_test_score')

In [ ]:
# (Automated tuning)

## Advanced Models

## Advanced Neural Networks

## Evaluation & Comparison

### Testing

#### Combined Results Table

### Confusion Matrices & Additional Plots

### Conclusion